##### This is for generating WordClouds based on the NGRAMS API and using Z-scores

In [36]:
#pip install stopwordsiso

In [37]:
#importing libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import time
from stopwordsiso import stopwords
import re

In [38]:
#loading the data

df = pd.read_csv('evaluation_results3.csv')
df.head(5)

,ID,Data Provider,Project Name,Consumer Team,Consumer Name,Consumer Description,Variation Type,Variation Value,Purpose,Realistic?,Decision,AI Decision 1,AI Decision 2,AI Decision 3,AI Warning 1,AI Warning 2,AI Warning 3,Vote Count,Final AI Decision
0,8392a6d9159cc8a43f7794b73694e375edddcaecc8afb2...,Benefit,Claims Risk Pattern Analysis,underwriting,Risk Insight Tool,A platform designed to offer in-depth risk ins...,NaN,Original Request,"To improve our risk assessment process, we aim...","Yes, this is a realistic access request for a ...",Accept,Accept,Accept,Accept,NaN,NaN,NaN,3-0,Accept
1,8392a6d9159cc8a43f7794b73694e375edddcaecc8afb2...,Benefit,Claims Risk Pattern Analysis,underwriting,Risk Insight Tool,A platform designed to offer in-depth risk ins...,combined,"intern + very hasty (typos, shorthand, missing...",hey need access to old claims data asap gotta ...,NaN,NaN,Reject,Reject,Reject,"{'title': 'Default policies', 'policyKey': 'gl...","{'policyKey': 'Default policies', 'title': 'Da...","{'policyKey': 'default', 'title': 'Default pol...",0-3,Reject
2,8392a6d9159cc8a43f7794b73694e375edddcaecc8afb2...,Benefit,Claims Risk Pattern Analysis,underwriting,Risk Insight Tool,A platform designed to offer in-depth risk ins...,combined,"intern + neutral (standard professional, no pa...",I'm looking to access historical claims data t...,NaN,NaN,Accept,Accept,Accept,NaN,NaN,NaN,3-0,Accept
3,8392a6d9159cc8a43f7794b73694e375edddcaecc8afb2...,Benefit,Claims Risk Pattern Analysis,underwriting,Risk Insight Tool,A platform designed to offer in-depth risk ins...,combined,intern + very formal (precise legal-style lang...,I hereby formally request access to the histor...,NaN,NaN,Accept,Accept,Accept,NaN,NaN,NaN,3-0,Accept
4,8392a6d9159cc8a43f7794b73694e375edddcaecc8afb2...,Benefit,Claims Risk Pattern Analysis,underwriting,Risk Insight Tool,A platform designed to offer in-depth risk ins...,combined,"junior analyst + very hasty (typos, shorthand,...",hey can I get access to the old claims data? n...,NaN,NaN,Reject,Reject,Reject,"{'policyKey': 'Default policies', 'title': 'Da...","{'policyKey': 'Default policies', 'title': 'Da...","{'policyKey': 'Default policies', 'title': 'Da...",0-3,Reject


In [39]:
#extracting columns for the seniority and the hastiness dimension.

def extracting_seniority(text):
    text = str(text).lower()

    if "intern" in text:
        return "Intern"
    elif "junior analyst" in text:
        return "Junior Analyst"
    elif "senior manager" in text:
        return "Senior Manager"
    elif "executive/ceo" in text:
        return "Executive/CEO"
    #elif "original request" in text:
    #    return "Original"
    return None


def extracting_hastiness(text):
    text = str(text).lower()

    if "very hasty" in text:
        return "Very Hasty"
    elif "neutral" in text:
        return "Neutral"
    elif "very formal" in text:
        return "Very Formal"
    #elif "original request" in text:
        #return "Original"
    return None

df['Seniority'] = df['Variation Value'].apply(extracting_seniority)
df['Hastiness'] = df['Variation Value'].apply(extracting_hastiness)



seniority_category=["Intern", "Junior Analyst", "Senior Manager", "Executive/CEO", "Original"]
hastiness_category=["Very Hasty", "Neutral", "Very Formal", "Original"]

df["Seniority"] = pd.Categorical(df["Seniority"],categories = seniority_category, ordered=True)
df["Hastiness"] = pd.Categorical(df["Hastiness"],categories = hastiness_category, ordered=True)



In [40]:
#cleaning the text and tokenising it (removing punctuation etc.)

def tokenise(text):
    if pd.isna(text):
        return []
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", "", text) #removing any punctuation from the prompt
    text = re.sub(r"http\S+", "", text) #removing any URLs from the prompt if there's any
    text = re.sub(r"\s+", " ", text) #removing any extra whitespace

    words = text.split() #splitting the text into words

    return words

#applying the tokenis into the purpose column
df["Tokens"] = (df["Purpose"].apply(tokenise))

#checking if it works
df[['Purpose', 'Tokens']].head(5)

,Purpose,Tokens
0,"To improve our risk assessment process, we aim...","[to, improve, our, risk, assessment, process, ..."
1,hey need access to old claims data asap gotta ...,"[hey, need, access, to, old, claims, data, asa..."
2,I'm looking to access historical claims data t...,"[im, looking, to, access, historical, claims, ..."
3,I hereby formally request access to the histor...,"[i, hereby, formally, request, access, to, the..."
4,hey can I get access to the old claims data? n...,"[hey, can, i, get, access, to, the, old, claim..."


In [41]:
print(df["Tokens"].iloc[1])

['hey', 'need', 'access', 'to', 'old', 'claims', 'data', 'asap', 'gotta', 'check', 'out', 'risk', 'patterns', 'tied', 'to', 'diff', 'groups', 'and', 'actions', 'wanna', 'use', 'beneficiaries', 'info', 'to', 'make', 'risk', 'profiles', 'for', 'various', 'customer', 'types', 'thisll', 'help', 'us', 'tweak', 'underwriting', 'and', 'predict', 'premiums', 'better', 'for', 'upcoming', 'policies', 'goal', 'is', 'to', 'make', 'sure', 'our', 'policies', 'match', 'real', 'risk', 'so', 'it', 'helps', 'the', 'company', 'with', 'better', 'financial', 'guesses', 'and', 'gives', 'customers', 'fair', 'insurance', 'prices', 'thx']


In [44]:
#calculating the word frequencies, using the Counter library to count the words

def get_word_frequencies(words):
    all_words = []

    for token in words:
        all_words.extend(token)
    word_counts = Counter(all_words)

    frequency_df = pd.DataFrame(word_counts.items(), columns=["Word", "Count"])

    total_words = frequency_df["Count"].sum()
    frequency_df["Relative Frequency"] = (frequency_df["Count"] / total_words)

    return frequency_df.sort_values(by="Count", ascending=False).reset_index(drop=True)



In [45]:
#applying the word frequency function to the different hastiness categories
very_hasty_words = df[df["Hastiness"] == "Very Hasty"]
very_formal_words = df[df["Hastiness"] == "Very Formal"]
neutral_words = df[df["Hastiness"] == "Neutral"]


very_hasty_word_freq = get_word_frequencies(very_hasty_words)
very_formal_word_freq = get_word_frequencies(very_formal_words)
neutral_word_freq = get_word_frequencies(neutral_words)


#displaying a table of the common words in the different hastiness categories
display(very_hasty_word_freq.head(10))
print()


,Word,Count,Relative Frequency
0,i,26,0.102767
1,e,22,0.086957
2,,22,0.086957
3,n,22,0.086957
4,o,18,0.071146
5,a,16,0.063241
6,s,15,0.059289
7,r,14,0.055336
8,t,10,0.039526
9,D,8,0.031621
